In [134]:
import os
from pyspark.sql import SparkSession

#os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
#os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark = SparkSession.builder \
    .appName("GarbageStats") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

In [135]:
df_train = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/train")

df_test = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/test")

In [136]:
from pyspark.sql.functions import lit, element_at, size, count, when, col, min, max, avg, split

In [137]:
df_train = df_train.withColumn("split", lit("train"))
df_test = df_test.withColumn("split", lit("test"))

df_raw = df_train.unionByName(df_test)

df_raw = df_raw.withColumn(
    "category",
    element_at(split("path", "/"), -2)
)

In [138]:
df_raw.count()

10464

In [139]:
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

+----+----------------+------+-------+-----+--------+
|path|modificationTime|length|content|split|category|
+----+----------------+------+-------+-----+--------+
|   0|               0|     0|      0|    0|       0|
+----+----------------+------+-------+-----+--------+



In [140]:
df_raw.groupBy("split").count().show()

+-----+-----+
|split|count|
+-----+-----+
|train| 7324|
| test| 3140|
+-----+-----+



In [141]:
df_raw.groupBy("category").count().orderBy("count", ascending=False).show()

+-------------+-----+
|     category|count|
+-------------+-----+
|        glass| 2649|
|biodegradable| 2220|
|        paper| 1746|
|    cardboard| 1439|
|        metal| 1248|
|      plastic| 1162|
+-------------+-----+



In [142]:
df_raw.select("length").describe().show()

+-------+------------------+
|summary|            length|
+-------+------------------+
|  count|             10464|
|   mean|19151.321769877675|
| stddev| 9474.638532461628|
|    min|              4188|
|    max|             67006|
+-------+------------------+



In [143]:
df_raw.groupBy("path").count().filter("count > 1").show() #doublons

+----+-----+
|path|count|
+----+-----+
+----+-----+



# DATA Transformées

In [144]:
df_train = spark.read.parquet("../data/train_data.parquet")
df_test = spark.read.parquet("../data/test_data.parquet")

In [145]:
df_train.printSchema()
df_test.printSchema()

root
 |-- x_train: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_train: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)

root
 |-- x_test: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_test: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)



In [146]:
df_train.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_train.columns
]).show()

+-------+-------+-----+
|x_train|y_train|class|
+-------+-------+-----+
|      0|      0|    0|
+-------+-------+-----+



In [147]:
df_test.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_test.columns
]).show()

+------+------+-----+
|x_test|y_test|class|
+------+------+-----+
|     0|     0|    0|
+------+------+-----+



In [148]:
df_train.groupBy("class").count().orderBy("count", ascending=False).show()
df_test.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass| 1854|
|biodegradable| 1554|
|        paper| 1223|
|    cardboard| 1006|
|        metal|  874|
|      plastic|  813|
+-------------+-----+

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass|  795|
|biodegradable|  666|
|        paper|  523|
|    cardboard|  433|
|        metal|  374|
|      plastic|  349|
+-------------+-----+



In [149]:
df_train = df_train.withColumn("height", size("x_train"))
df_train = df_train.withColumn("width", size(col("x_train")[0]))

df_train.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 7324|
+------+-----+-----+



In [150]:
df_test = df_test.withColumn("height", size("x_test"))
df_test = df_test.withColumn("width", size(col("x_test")[0]))

df_test.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 3140|
+------+-----+-----+



In [151]:
df_train.withColumn("label_size", size("y_train")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 7324|
+----------+-----+



In [152]:
df_test.withColumn("label_size", size("y_test")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 3140|
+----------+-----+



# Prédiction DATA

In [154]:
df_pred = spark.read.parquet("../data/prediction_data.parquet")

In [155]:
df_pred.count()

639

In [156]:
df_pred.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass|  132|
|        paper|  127|
|      plastic|  127|
|    cardboard|  112|
|        metal|  107|
|biodegradable|   34|
+-------------+-----+



In [157]:
df_pred.groupBy("file").count().filter("count > 1").show()

+-------------+-----+
|         file|count|
+-------------+-----+
|plastic21.jpg|    2|
|plastic36.jpg|    2|
|plastic35.jpg|    2|
|plastic29.jpg|    2|
|plastic12.jpg|    2|
|plastic27.jpg|    2|
|plastic32.jpg|    2|
|plastic44.jpg|    2|
|plastic31.jpg|    2|
|plastic26.jpg|    2|
|plastic18.jpg|    2|
|plastic24.jpg|    2|
|plastic13.jpg|    2|
|plastic20.jpg|    2|
|plastic30.jpg|    2|
|plastic37.jpg|    2|
|plastic22.jpg|    2|
|plastic15.jpg|    2|
|plastic19.jpg|    2|
|plastic33.jpg|    2|
+-------------+-----+
only showing top 20 rows


In [158]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).show()

+----+----------+--------+-----+----------+
|file|prediction|class_id|class|confidence|
+----+----------+--------+-----+----------+
|   0|         0|       0|    0|         0|
+----+----------+--------+-----+----------+



In [159]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .show()

+-------------+-----+------------------+
|        class|count|        percentage|
+-------------+-----+------------------+
|        glass|  132|20.657276995305164|
|        paper|  127|19.874804381846637|
|      plastic|  127|19.874804381846637|
|    cardboard|  112| 17.52738654147105|
|        metal|  107| 16.74491392801252|
|biodegradable|   34| 5.320813771517996|
+-------------+-----+------------------+



In [160]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .show()

+-------------+------------------+
|        class|    avg_confidence|
+-------------+------------------+
|    cardboard|0.6074149256039943|
|        paper|0.5666258121569325|
|      plastic| 0.557166252666571|
|        glass|0.4560071490705013|
|        metal|0.3560712900117179|
|biodegradable|0.2873006423606592|
+-------------+------------------+



In [161]:
df_pred.filter(col("confidence") < 0.5).show()

+-------------+--------------------+--------+-------------+-------------------+
|         file|          prediction|class_id|        class|         confidence|
+-------------+--------------------+--------+-------------+-------------------+
|  paper79.jpg|[0.05858080089092...|       5|        paper| 0.4362509250640869|
|  glass75.jpg|[0.01232929807156...|       3|        glass| 0.4934532642364502|
|  glass12.jpg|[0.16704802215099...|       4|        metal| 0.2654241919517517|
|plastic29.jpg|[0.30113026499748...|       5|        paper| 0.3295934498310089|
|  paper34.jpg|[0.10291779786348...|       5|        paper| 0.2711322605609894|
|  paper21.jpg|[0.01727584004402...|       3|        glass|0.36399200558662415|
|   glass7.jpg|[0.00376888969913...|       6|      plastic|0.46028026938438416|
|  glass23.jpg|[0.01992154866456...|       6|      plastic| 0.3557801842689514|
|plastic12.jpg|[0.01694130338728...|       5|        paper|0.39191409945487976|
|  paper26.jpg|[0.12414984405040...|    

In [162]:
df_pred.select("confidence").describe().show()

+-------+-------------------+
|summary|         confidence|
+-------+-------------------+
|  count|                639|
|   mean|0.49892465954468657|
| stddev|0.23351382887830208|
|    min|0.17791776359081268|
|    max|  0.999966025352478|
+-------+-------------------+



In [163]:
df_pred.orderBy("confidence", ascending=False).show(10)

+----------------+--------------------+--------+---------+------------------+
|            file|          prediction|class_id|    class|        confidence|
+----------------+--------------------+--------+---------+------------------+
|cardboard170.jpg|[2.33894396906020...|       2|cardboard| 0.999966025352478|
|cardboard257.jpg|[7.16508232953549...|       2|cardboard|0.9999039173126221|
|cardboard133.jpg|[8.35996716297415...|       2|cardboard|0.9998824596405029|
|cardboard279.jpg|[6.07658847684433...|       2|cardboard|0.9996837377548218|
|cardboard127.jpg|[3.0086631852555E...|       2|cardboard|0.9996474981307983|
|  plastic313.jpg|[7.41114472308651...|       2|cardboard| 0.999238133430481|
|cardboard136.jpg|[1.68636304920255...|       2|cardboard|0.9987162351608276|
|cardboard268.jpg|[6.01541159994667...|       2|cardboard|0.9986557960510254|
|cardboard259.jpg|[4.63735538858145...|       2|cardboard|0.9980636239051819|
|cardboard163.jpg|[1.59915565234314...|       6|  plastic|0.9968

# Ecriture parquet 

In [164]:
df_pred.select(count("*").alias("total")) \
    .write.mode("overwrite") \
    .parquet("../data/stats/total.parquet")

In [165]:
df_pred.groupBy("class") \
    .count() \
    .orderBy("count", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/count_by_class.parquet")

In [166]:
df_pred.groupBy("file") \
    .count() \
    .filter("count > 1") \
    .write.mode("overwrite") \
    .parquet("../data/stats/duplicates.parquet")

In [167]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).write.mode("overwrite") \
 .parquet("../data/stats/nulls.parquet")

In [168]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/percentage_by_class.parquet")

In [169]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_by_class.parquet")

In [170]:
df_pred.filter(col("confidence") < 0.5) \
    .write.mode("overwrite") \
    .parquet("../data/stats/low_confidence.parquet")

In [171]:
train_count = df_train.count()
test_count = df_test.count()

total = train_count + test_count

print("Train :", train_count)
print("Test :", test_count)
print("Total :", total)

Train : 7324
Test : 3140
Total : 10464


In [172]:
df_pred.select("confidence") \
    .describe() \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_stats.parquet")

In [173]:
df_pred.orderBy("confidence", ascending=False) \
    .limit(10) \
    .write.mode("overwrite") \
    .parquet("../data/stats/top_predictions.parquet")